# Stages 3+4 — measurement suite & unlearning (Colab T4 driver)

Part of **"Unlearning that isn't"**. Requires the Stage 2 checkpoint in Drive at `MyDrive/unlearn/checkpoints/memorized` (the Stage 2 notebook put it there).

Runs: Stage 3 (exposure, Min-K% MIA with never-trained controls, natural-text utility, frequency curve) then Stage 4 (GA / GD / NPO unlearning, each gated).

Runtime → **T4 GPU**. Expected total: ~20–30 min (most of it the three unlearning runs).

In [ ]:
GIST_ID = "0a08a15903ed0b8679086f875e223fa0"

import pathlib, shutil, subprocess
subprocess.run(["git", "clone", "-q", f"https://gist.github.com/{GIST_ID}.git", "gistsrc"], check=True)
root = pathlib.Path("llm-unlearning-reality-check")
for p in pathlib.Path("gistsrc").iterdir():
    if p.name.startswith(".") or p.suffix == ".ipynb":
        continue
    dest = root / p.name.replace("___", "/", 1)
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(p, dest)
%cd llm-unlearning-reality-check
%pip -q install -r requirements.txt

In [ ]:
# restore the memorized checkpoint from Drive; regenerate the (deterministic) dataset
from google.colab import drive
drive.mount("/content/drive")
!cp -r /content/drive/MyDrive/unlearn/checkpoints .
!ls checkpoints/memorized
!python -m src.canaries --config configs/memorize.yaml | tail -3

## Stage 3 — measurement suite

In [ ]:
!python -m src.eval_memorization --config configs/eval.yaml
from IPython.display import Image
Image("results/freq_curve.png")

## Stage 4 — unlearning, three methods

Each starts from the same memorized checkpoint. GA is expected to struggle (that's a finding); GD and NPO should pass.

In [ ]:
!python -m src.unlearn --config configs/unlearn_ga.yaml

In [ ]:
!python -m src.unlearn --config configs/unlearn_gd.yaml

In [ ]:
!python -m src.unlearn --config configs/unlearn_npo.yaml

In [ ]:
import json
for m in ("ga", "gd", "npo"):
    try:
        d = json.load(open(f"results/unlearn_{m}.json"))["data"]
        print(f"{m:4s} gate={d['gate']} steps={d['steps_run']} rollbacks={d['rollbacks']} "
              f"ppl={d['ppl_natural']:.1f} overforgotten={[r['id'] for r in d['table'] if r['overforgotten']]}")
    except FileNotFoundError:
        print(m, "missing")

## Persist outputs (checkpoints -> Drive, results -> download)

In [ ]:
from google.colab import files
!cp -r checkpoints /content/drive/MyDrive/unlearn/
!zip -qr results_stage34.zip results
files.download("results_stage34.zip")